# 🔄 Step 2: Data Preprocessing

This notebook preprocesses the LFW images for training:
- Resize images to 112x112 (optimal for lightweight models)
- Normalize pixel values
- Create train/val/test splits
- Build PyTorch datasets

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import cv2
import pickle
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# Add project root
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Paths
RAW_DIR = PROJECT_ROOT / "data" / "lfw" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "lfw" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data: {RAW_DIR}")
print(f"Output: {PROCESSED_DIR}")

## 2.1 Load Raw Data

In [ ]:
# Load raw data
print("Loading raw data...")

images = np.load(RAW_DIR / "images.npy")
labels = np.load(RAW_DIR / "labels.npy")
names = np.load(RAW_DIR / "names.npy")

print(f"✅ Loaded {len(images)} images of {len(names)} people")
print(f"   Original shape: {images[0].shape}")

## 2.2 Preprocess Images

**For a lightweight, fast model:**
- Use **112x112** instead of 160x160 (faster inference)
- Normalize to **[-1, 1]** (standard for face models)

In [ ]:
# Configuration for lightweight model
TARGET_SIZE = 112  # Smaller = faster (112 is common for mobile face models)

def preprocess_image(img, target_size=TARGET_SIZE):
    """Preprocess a single image."""
    # Convert to uint8 if float
    if img.max() <= 1.0:
        img = (img * 255).astype(np.uint8)
    
    # Resize
    resized = cv2.resize(img, (target_size, target_size))
    
    # Normalize to [-1, 1]
    normalized = (resized.astype(np.float32) - 127.5) / 127.5
    
    return normalized

# Process all images
print(f"Preprocessing {len(images)} images to {TARGET_SIZE}x{TARGET_SIZE}...")

processed_images = []
for img in tqdm(images):
    processed = preprocess_image(img)
    processed_images.append(processed)

processed_images = np.array(processed_images)
print(f"✅ Processed shape: {processed_images.shape}")
print(f"   Pixel range: [{processed_images.min():.2f}, {processed_images.max():.2f}]")

In [ ]:
# Visualize preprocessing
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i in range(5):
    idx = np.random.randint(0, len(images))
    
    # Original
    orig = images[idx]
    if orig.max() > 1:
        orig = orig / 255.0
    axes[0, i].imshow(orig)
    axes[0, i].set_title(f'Original\n{images[idx].shape}')
    axes[0, i].axis('off')
    
    # Processed
    proc = processed_images[idx]
    proc_display = (proc + 1) / 2  # Convert [-1,1] to [0,1] for display
    axes[1, i].imshow(proc_display)
    axes[1, i].set_title(f'Processed\n{proc.shape}')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('Processed', fontsize=12)
plt.suptitle('Image Preprocessing', fontsize=14)
plt.tight_layout()
plt.show()

## 2.3 Create Train/Val/Test Splits

In [ ]:
# Stratified split
VAL_RATIO = 0.15
TEST_RATIO = 0.15

print(f"Creating splits: {(1-VAL_RATIO-TEST_RATIO)*100:.0f}% train, {VAL_RATIO*100:.0f}% val, {TEST_RATIO*100:.0f}% test")

# First split: train+val vs test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    processed_images, labels,
    test_size=TEST_RATIO,
    stratify=labels,
    random_state=42
)

# Second split: train vs val
val_size_adjusted = VAL_RATIO / (1 - TEST_RATIO)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=val_size_adjusted,
    stratify=y_trainval,
    random_state=42
)

print(f"\n✅ Split complete:")
print(f"   Train: {len(X_train)} images ({len(X_train)/len(processed_images)*100:.1f}%)")
print(f"   Val:   {len(X_val)} images ({len(X_val)/len(processed_images)*100:.1f}%)")
print(f"   Test:  {len(X_test)} images ({len(X_test)/len(processed_images)*100:.1f}%)")

## 2.4 Save Processed Data

In [ ]:
# Save processed data
print("Saving processed data...")

np.save(PROCESSED_DIR / "train_images.npy", X_train)
np.save(PROCESSED_DIR / "train_labels.npy", y_train)
np.save(PROCESSED_DIR / "val_images.npy", X_val)
np.save(PROCESSED_DIR / "val_labels.npy", y_val)
np.save(PROCESSED_DIR / "test_images.npy", X_test)
np.save(PROCESSED_DIR / "test_labels.npy", y_test)
np.save(PROCESSED_DIR / "label_names.npy", names)

# Save metadata
metadata = {
    "train_size": len(X_train),
    "val_size": len(X_val),
    "test_size": len(X_test),
    "num_classes": len(names),
    "image_size": TARGET_SIZE,
    "image_shape": X_train[0].shape,
}

with open(PROCESSED_DIR / "metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

print(f"\n✅ Data saved to: {PROCESSED_DIR}")
for f in PROCESSED_DIR.glob("*"):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"   {f.name}: {size_mb:.1f} MB")

## 2.5 Process Verification Pairs

In [ ]:
# Load and process verification pairs
pairs = np.load(RAW_DIR / "pairs.npy")
pairs_labels = np.load(RAW_DIR / "pairs_labels.npy")

print(f"Processing {len(pairs)} verification pairs...")

processed_pairs = []
for pair in tqdm(pairs):
    img1 = preprocess_image(pair[0])
    img2 = preprocess_image(pair[1])
    processed_pairs.append([img1, img2])

processed_pairs = np.array(processed_pairs)

np.save(PROCESSED_DIR / "test_pairs.npy", processed_pairs)
np.save(PROCESSED_DIR / "test_pairs_labels.npy", pairs_labels)

print(f"✅ Saved {len(processed_pairs)} processed pairs")

## ✅ Step 2 Complete!

**Summary:**
- Preprocessed images to 112x112 for lightweight model
- Created stratified train/val/test splits
- Saved processed data to `data/lfw/processed/`

**👉 Next Step:** Run `03_model_architecture.ipynb` to build the model